In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').getOrCreate()

In [67]:
# Import all functions
from pyspark.sql.functions import *

In [7]:
data = [('Alicia', 'Joseph', ['Java', 'Scala', 'Spark'], {'hair': 'black', 'eye': 'brown'}),
        ('Robert', 'Gee', ['Spark', 'Java'], {'hair': 'brown', 'eye': None}),
        ('Mike', 'Bianca', ['CSharp', ''], {'hair': 'red', 'eye': ''}),
        ('John', 'Kumar', None, None),
        ('Jeff', 'L', ['1', '2'], {})]

schema = ['FirstName', 'LastName', 'Languages', 'Properties']

emp1 = spark.createDataFrame(data, schema)
emp1.show(truncate=False)
emp1.printSchema()

+---------+--------+--------------------+-----------------------------+
|FirstName|LastName|Languages           |Properties                   |
+---------+--------+--------------------+-----------------------------+
|Alicia   |Joseph  |[Java, Scala, Spark]|{eye -> brown, hair -> black}|
|Robert   |Gee     |[Spark, Java]       |{eye -> NULL, hair -> brown} |
|Mike     |Bianca  |[CSharp, ]          |{eye -> , hair -> red}       |
|John     |Kumar   |NULL                |NULL                         |
|Jeff     |L       |[1, 2]              |{}                           |
+---------+--------+--------------------+-----------------------------+

root
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Languages: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- Properties: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)



### `size(col)`
- Returns length of the array or map stored in the column
- Size is NULL for NULL elements

In [8]:
emp1.select('Languages', size('Languages'), 'Properties', size('Properties')).show(truncate=False)

+--------------------+---------------+-----------------------------+----------------+
|Languages           |size(Languages)|Properties                   |size(Properties)|
+--------------------+---------------+-----------------------------+----------------+
|[Java, Scala, Spark]|3              |{eye -> brown, hair -> black}|2               |
|[Spark, Java]       |2              |{eye -> NULL, hair -> brown} |2               |
|[CSharp, ]          |2              |{eye -> , hair -> red}       |2               |
|NULL                |NULL           |NULL                         |NULL            |
|[1, 2]              |2              |{}                           |0               |
+--------------------+---------------+-----------------------------+----------------+



### `element_at(col, extraction)`
- If array, returns element of array at given (1-based) index. If Index is 0, Spark will throw an error.

- If map, returns value for given key in extraction. Returns NULL if the key is not contained in the map.

In [9]:
emp1.select('Languages', element_at('Languages', 2), 'Properties', element_at('Properties', 'eye')).show(truncate=False)

+--------------------+------------------------+-----------------------------+---------------------------+
|Languages           |element_at(Languages, 2)|Properties                   |element_at(Properties, eye)|
+--------------------+------------------------+-----------------------------+---------------------------+
|[Java, Scala, Spark]|Scala                   |{eye -> brown, hair -> black}|brown                      |
|[Spark, Java]       |Java                    |{eye -> NULL, hair -> brown} |NULL                       |
|[CSharp, ]          |                        |{eye -> , hair -> red}       |                           |
|NULL                |NULL                    |NULL                         |NULL                       |
|[1, 2]              |2                       |{}                           |NULL                       |
+--------------------+------------------------+-----------------------------+---------------------------+



### `struct(cols)`
- Creates a struct type column using the columns passed in the argument

In [10]:
struct_df = emp1.select(struct('FirstName', 'LastName').alias('new_struct_col'))
struct_df.show()
struct_df.printSchema()

+----------------+
|  new_struct_col|
+----------------+
|{Alicia, Joseph}|
|   {Robert, Gee}|
|  {Mike, Bianca}|
|   {John, Kumar}|
|       {Jeff, L}|
+----------------+

root
 |-- new_struct_col: struct (nullable = false)
 |    |-- FirstName: string (nullable = true)
 |    |-- LastName: string (nullable = true)



### `array(cols)`
- Creates an array type column using the columns passed in the argument

In [11]:
array_df = emp1.select(array('FirstName', 'LastName').alias('new_array_col'))
array_df.show()
array_df.printSchema()

+----------------+
|   new_array_col|
+----------------+
|[Alicia, Joseph]|
|   [Robert, Gee]|
|  [Mike, Bianca]|
|   [John, Kumar]|
|       [Jeff, L]|
+----------------+

root
 |-- new_array_col: array (nullable = false)
 |    |-- element: string (containsNull = true)



### `create_map(cols)`
- Creates a new map column
- It requires even number of input columns so that 1st column will be key and 2nd will be value and so on

In [39]:
emp1.select(create_map('FirstName', 'Lastname')).show()
emp1.select(create_map('FirstName', 'Lastname')).printSchema()

+------------------------+
|map(FirstName, Lastname)|
+------------------------+
|      {Alicia -> Joseph}|
|         {Robert -> Gee}|
|        {Mike -> Bianca}|
|         {John -> Kumar}|
|             {Jeff -> L}|
+------------------------+

root
 |-- map(FirstName, Lastname): map (nullable = false)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)



In [12]:
data = [('John', [10, 20, 20], [25, 11, 10]), ('Robert', [15, 13, 55], [5, None, 29]), ('James', [11, 13, 45], [5, 89, 79])]
schema = ['empName', 'score_arr1', 'score_arr2']

emp3 = spark.createDataFrame(data, schema)
emp3.show(truncate=False)
emp3.printSchema()

+-------+------------+-------------+
|empName|score_arr1  |score_arr2   |
+-------+------------+-------------+
|John   |[10, 20, 20]|[25, 11, 10] |
|Robert |[15, 13, 55]|[5, NULL, 29]|
|James  |[11, 13, 45]|[5, 89, 79]  |
+-------+------------+-------------+

root
 |-- empName: string (nullable = true)
 |-- score_arr1: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- score_arr2: array (nullable = true)
 |    |-- element: long (containsNull = true)



### `array_max(col)`
- Returns the maximum value of the array

###`array_distinct(col)`
- Returns the distinct values from the array

In [13]:
emp3.select('score_arr1', array_max('score_arr1').alias('score_arr1_max'), array_distinct('score_arr1').alias('score_arr1_distinct')).show()

+------------+--------------+-------------------+
|  score_arr1|score_arr1_max|score_arr1_distinct|
+------------+--------------+-------------------+
|[10, 20, 20]|            20|           [10, 20]|
|[15, 13, 55]|            55|       [15, 13, 55]|
|[11, 13, 45]|            45|       [11, 13, 45]|
+------------+--------------+-------------------+



### `array_repeat(col, count)`
- Creates an array containing a column repeated `count` times.

In [14]:
emp3.select(array_repeat('empName', 3).alias('repeat_array')).show(truncate=False)

+------------------------+
|repeat_array            |
+------------------------+
|[John, John, John]      |
|[Robert, Robert, Robert]|
|[James, James, James]   |
+------------------------+



### `slice(col, start, length)`
- Returns a new array column by slicing the input array column from
a `start` index to a specific `length`.
- The indices start at 1.
- The `length` specifies the number of elements in the resulting array.

In [15]:
# Slice to start from index = 2 to length = 2
emp3.select('score_arr1', slice('score_arr1', 2, 2).alias('sliced_col')).show(truncate=False)

+------------+----------+
|score_arr1  |sliced_col|
+------------+----------+
|[10, 20, 20]|[20, 20]  |
|[15, 13, 55]|[13, 55]  |
|[11, 13, 45]|[13, 45]  |
+------------+----------+



In [16]:
# Slice to start from index = -1 to length = 2
emp3.select('score_arr1', slice('score_arr1', -1, 2).alias('sliced_col')).show(truncate=False)

+------------+----------+
|score_arr1  |sliced_col|
+------------+----------+
|[10, 20, 20]|[20]      |
|[15, 13, 55]|[55]      |
|[11, 13, 45]|[45]      |
+------------+----------+



### `array_position(col, value)`
- Locates the position of the first occurrence of the given value
in the given array.
- Returns 0 if not found.

In [17]:
emp3.select('score_arr1', array_position('score_arr1', 13).alias('position_of_13')).show(truncate=False)

+------------+--------------+
|score_arr1  |position_of_13|
+------------+--------------+
|[10, 20, 20]|0             |
|[15, 13, 55]|2             |
|[11, 13, 45]|2             |
+------------+--------------+



### `array_remove(col, element)`
- Remove all elements that equal to element from the given array.

In [18]:
emp3.select('score_arr1', array_remove('score_arr1', 13).alias('after_removed')).show(truncate=False)

+------------+-------------+
|score_arr1  |after_removed|
+------------+-------------+
|[10, 20, 20]|[10, 20, 20] |
|[15, 13, 55]|[15, 55]     |
|[11, 13, 45]|[11, 45]     |
+------------+-------------+



### `array_sort(col)`
- Sorts the input array in ascending order.
- NULL elements will be placed at the end of the returned array

In [19]:
emp3.select('score_arr2', array_sort('score_arr2').alias('sorted')).show(truncate=False)

+-------------+-------------+
|score_arr2   |sorted       |
+-------------+-------------+
|[25, 11, 10] |[10, 11, 25] |
|[5, NULL, 29]|[5, 29, NULL]|
|[5, 89, 79]  |[5, 79, 89]  |
+-------------+-------------+



### `sort_array(col, asc)`
- Sorts the input array in ascending or descending order according to the natural ordering of the array elements.
- NULL elements will be placed at the beginning if ascending  or at the end if descending.

In [20]:
emp3.select('score_arr2', sort_array('score_arr2', asc=True).alias('sorted')).show(truncate=False)

+-------------+-------------+
|score_arr2   |sorted       |
+-------------+-------------+
|[25, 11, 10] |[10, 11, 25] |
|[5, NULL, 29]|[NULL, 5, 29]|
|[5, 89, 79]  |[5, 79, 89]  |
+-------------+-------------+



In [21]:
emp3.select('score_arr2', sort_array('score_arr2', asc=False).alias('sorted')).show(truncate=False)

+-------------+-------------+
|score_arr2   |sorted       |
+-------------+-------------+
|[25, 11, 10] |[25, 11, 10] |
|[5, NULL, 29]|[29, 5, NULL]|
|[5, 89, 79]  |[89, 79, 5]  |
+-------------+-------------+



### `array_contains(col, value)`
- Returns True if value is present else False
- Returns NULL if array has NULL values

In [22]:
emp3.select('score_arr2', array_contains('score_arr2', 11).alias('check')).show(truncate=False)

+-------------+-----+
|score_arr2   |check|
+-------------+-----+
|[25, 11, 10] |true |
|[5, NULL, 29]|NULL |
|[5, 89, 79]  |false|
+-------------+-----+



### `array_union(col1, col2)`
- Returns a new array containing the union of elements in col1 and col2, without duplicates.
- Takes only 2 column arguments at a time

In [23]:
emp3.select('score_arr1', 'score_arr2', array_union('score_arr1', 'score_arr2').alias('score_union')).show(truncate=False)

+------------+-------------+-------------------------+
|score_arr1  |score_arr2   |score_union              |
+------------+-------------+-------------------------+
|[10, 20, 20]|[25, 11, 10] |[10, 20, 25, 11]         |
|[15, 13, 55]|[5, NULL, 29]|[15, 13, 55, 5, NULL, 29]|
|[11, 13, 45]|[5, 89, 79]  |[11, 13, 45, 5, 89, 79]  |
+------------+-------------+-------------------------+



### `array_except(col1, col2)`
- Returns a new array containing the elements present in col1 but not in col2,
without duplicates.
- Takes only 2 column arguments at a time

In [24]:
emp3.select('score_arr1', 'score_arr2', array_except('score_arr1', 'score_arr2').alias('score_except')).show(truncate=False)

+------------+-------------+------------+
|score_arr1  |score_arr2   |score_except|
+------------+-------------+------------+
|[10, 20, 20]|[25, 11, 10] |[20]        |
|[15, 13, 55]|[5, NULL, 29]|[15, 13, 55]|
|[11, 13, 45]|[5, 89, 79]  |[11, 13, 45]|
+------------+-------------+------------+



### `array_intersect(col1, col2)`
- Returns a new array containing the intersection of elements in col1 and col2,
without duplicates.
- Takes only 2 column arguments at a time

In [25]:
emp3.select('score_arr1', 'score_arr2', array_intersect('score_arr1', 'score_arr2').alias('score_intersect')).show(truncate=False)

+------------+-------------+---------------+
|score_arr1  |score_arr2   |score_intersect|
+------------+-------------+---------------+
|[10, 20, 20]|[25, 11, 10] |[10]           |
|[15, 13, 55]|[5, NULL, 29]|[]             |
|[11, 13, 45]|[5, 89, 79]  |[]             |
+------------+-------------+---------------+



### `array_join(col, delimiter, null_replacement)`
- Returns a string column by concatenating the elements of the input array column using the delimiter.
- NULL values within the array can be replaced with a specified string through the null_replacement argument.
- If null_replacement is not set, NULL values are ignored.

In [26]:
emp3.select('score_arr2', array_join('score_arr2', delimiter='|').alias('score_join')).show(truncate=False)

+-------------+----------+
|score_arr2   |score_join|
+-------------+----------+
|[25, 11, 10] |25|11|10  |
|[5, NULL, 29]|5|29      |
|[5, 89, 79]  |5|89|79   |
+-------------+----------+



In [27]:
emp3.select('score_arr2', array_join('score_arr2', delimiter='|', null_replacement='*').alias('score_join')).show(truncate=False)

+-------------+----------+
|score_arr2   |score_join|
+-------------+----------+
|[25, 11, 10] |25|11|10  |
|[5, NULL, 29]|5|*|29    |
|[5, 89, 79]  |5|89|79   |
+-------------+----------+



### `arrays_zip(cols)`
- Merges the array elements
- 1st element of array1 will be merged with 1st element of array2 and so on

In [30]:
emp3.select('score_arr1', 'score_arr2', arrays_zip('score_arr1', 'score_arr2').alias('zipped')).show(truncate=False)

+------------+-------------+-------------------------------+
|score_arr1  |score_arr2   |zipped                         |
+------------+-------------+-------------------------------+
|[10, 20, 20]|[25, 11, 10] |[{10, 25}, {20, 11}, {20, 10}] |
|[15, 13, 55]|[5, NULL, 29]|[{15, 5}, {13, NULL}, {55, 29}]|
|[11, 13, 45]|[5, 89, 79]  |[{11, 5}, {13, 89}, {45, 79}]  |
+------------+-------------+-------------------------------+



### `arrays_overlap(a1, a2)`
- Returns True if the arrays contain any common non-null element
- Returns NULL if both the arrays are non-empty and any one of them contains NULL element
- Else returns False

In [32]:
emp3.select('score_arr1', 'score_arr2', arrays_overlap('score_arr1', 'score_arr2').alias('overlap')).show(truncate=False)

+------------+-------------+-------+
|score_arr1  |score_arr2   |overlap|
+------------+-------------+-------+
|[10, 20, 20]|[25, 11, 10] |true   |
|[15, 13, 55]|[5, NULL, 29]|NULL   |
|[11, 13, 45]|[5, 89, 79]  |false  |
+------------+-------------+-------+



### `shuffle(col)`
- Randomly shuufles the array elements

In [37]:
emp3.select('score_arr2', shuffle('score_arr2').alias('shuffled')).show(truncate=False)

+-------------+-------------+
|score_arr2   |shuffled     |
+-------------+-------------+
|[25, 11, 10] |[11, 10, 25] |
|[5, NULL, 29]|[NULL, 5, 29]|
|[5, 89, 79]  |[89, 5, 79]  |
+-------------+-------------+



In [5]:
df = spark.sql("SELECT ARRAY(STRUCT(1, 'a), STRUCT(2, 'b)) AS data")
df.show(truncate=False)
df.printSchema()

+---------------------+
|data                 |
+---------------------+
|[{1, a), STRUCT(2, }]|
+---------------------+

root
 |-- data: array (nullable = false)
 |    |-- element: struct (containsNull = false)
 |    |    |-- col1: integer (nullable = false)
 |    |    |-- b: string (nullable = false)



### `map_from_entries(col)`
- Transforms an array of key-value pair entries (structs with two fields) into a map.
- The first field of each entry is used as the key and the second field
as the value

In [46]:
df.printSchema() # Before
df.select(map_from_entries('data').alias('mapped_data')).printSchema() # After

root
 |-- data: array (nullable = false)
 |    |-- element: struct (containsNull = false)
 |    |    |-- col1: integer (nullable = false)
 |    |    |-- b: string (nullable = false)

root
 |-- mapped_data: map (nullable = false)
 |    |-- key: integer
 |    |-- value: string (valueContainsNull = false)



### `map_from_arrays(col1, col2)`
- Creates a new map from two arrays.

In [53]:
df = spark.createDataFrame([([2, 5], ['a', 'b'])], ['k', 'v'])
df.show()
df.select(map_from_arrays(df.k, df.v)).show()

+------+------+
|     k|     v|
+------+------+
|[2, 5]|[a, b]|
+------+------+

+---------------------+
|map_from_arrays(k, v)|
+---------------------+
|     {2 -> a, 5 -> b}|
+---------------------+



### `map_keys(col)`
- Returns an array containing keys of the map column

In [57]:
emp1.select('Properties', map_keys('Properties').alias('keys')).show(truncate=False)

+-----------------------------+-----------+
|Properties                   |keys       |
+-----------------------------+-----------+
|{eye -> brown, hair -> black}|[eye, hair]|
|{eye -> NULL, hair -> brown} |[eye, hair]|
|{eye -> , hair -> red}       |[eye, hair]|
|NULL                         |NULL       |
|{}                           |[]         |
+-----------------------------+-----------+



### `map_values(col)`
- Returns an array containing values of the map column

In [58]:
emp1.select('Properties', map_values('Properties').alias('values')).show(truncate=False)

+-----------------------------+--------------+
|Properties                   |values        |
+-----------------------------+--------------+
|{eye -> brown, hair -> black}|[brown, black]|
|{eye -> NULL, hair -> brown} |[NULL, brown] |
|{eye -> , hair -> red}       |[, red]       |
|NULL                         |NULL          |
|{}                           |[]            |
+-----------------------------+--------------+



### `map_concat(cols)`
- Returns the union of all given maps

In [61]:
df = spark.sql("SELECT map(1, 'a', 2, 'b') as map1, map(3, 'c') as map2")
df.show()
df.select(map_concat("map1", "map2")).show(truncate=False)

+----------------+--------+
|            map1|    map2|
+----------------+--------+
|{1 -> a, 2 -> b}|{3 -> c}|
+----------------+--------+

+------------------------+
|map_concat(map1, map2)  |
+------------------------+
|{1 -> a, 2 -> b, 3 -> c}|
+------------------------+



### `sequence(start, stop, step)`
- Generate a sequence of integers from start to stop, incrementing by step.
- If step is not set, the function increments by 1 if start is less than or equal to stop, otherwise it decrements by 1.

In [65]:
df = spark.createDataFrame([(-2, 2)], ['start', 'stop'])
df.show()
df.select('start', 'stop', sequence('start', 'stop').alias('sequence')).show()

+-----+----+
|start|stop|
+-----+----+
|   -2|   2|
+-----+----+

+-----+----+-----------------+
|start|stop|         sequence|
+-----+----+-----------------+
|   -2|   2|[-2, -1, 0, 1, 2]|
+-----+----+-----------------+



In [66]:
df = spark.createDataFrame([(5, 1)], ['start', 'stop'])
df.show()
df.select('start', 'stop', sequence('start', 'stop').alias('sequence')).show()

+-----+----+
|start|stop|
+-----+----+
|    5|   1|
+-----+----+

+-----+----+---------------+
|start|stop|       sequence|
+-----+----+---------------+
|    5|   1|[5, 4, 3, 2, 1]|
+-----+----+---------------+

